In [3]:
#%% SIMPLIFIED EMISSION BUDGETS -- For regional and global budgets 
# can toggle ocean settings on and off when wanting marine ecosystems only
import numpy as np
import xarray as xr
from netCDF4 import Dataset
from pathlib import Path

# LOAD IN THE NETCDF FILE -----------------------------------------
# HIGH emissions 
#path = Path("/home/hplaas/nobackup/JupyterLinks/notebooks/cesm_files/output_files")
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL0.2-WOOD10-OIL38.cam.h1MEAN.nc" # HIGH - BASE
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL33-WOOD10-OIL38.cam.h1MEAN.nc" # HIGH - RESI 
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL33-WOOD56-OIL38.cam.h1MEAN.nc" # HIGH - WOOD 
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.05-RESICOAL33-WOOD56-OIL25.cam.h1_MEAN.nc" # HIGH - INDUSTRIAL

# Central-High emissions
#path = Path("/home/hplaas/nobackup/JupyterLinks/notebooks/cesm_files/output_files")
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL0.2-WOOD10-OIL38_scale_BF.cam.h1.2009-2011_filtered_MEAN.nc" # LOW - BASE
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL33-WOOD10-OIL38_scale_BF.cam.h1.2009-2011_MEAN.nc" # LOW - RESI 
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL33-WOOD56-OIL38_scale_BF_rerun_v2.cam.h1.2009-2011_MEAN.nc" # LOW - WOOD -- dust not scaled

# Central-Low
path = Path("/home/hplaas/nobackup/JupyterLinks/notebooks/cesm_files/output_files/MEDFEFRAC_reruns/DUST_FIX")
case = "BIOF" # BASE RESI WOOD IND
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL0.2-WOOD10-OIL38-WASTE1.5_MEDFEFRAC_DUSTFETUNED.cam.h1.2009-2011_import_DUST_MEAN.nc" # BASE
#sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL33-WOOD10-OIL38-WASTE1.5_MEDFEFRAC_DUSTFETUNED.cam.h1.2009-2011_import_DUST_MEAN.nc" # RESI 
sim_directory = f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.2-RESICOAL33-WOOD56-OIL38-WASTE1.5_MEDFEFRAC_DUSTFETUNED.cam.h1.2009-2011_import_DUST_MEAN.nc" # WOOD 
#sim_directory = f"{path}/f"{path}/CAM6-MIMI_2010CLIMO_INDCOAL0.05-RESICOAL33-WOOD56-OIL25-WASTE1.5_MEDFEFRAC_DUSTFETUNED.cam.h1.2009-2011_MEAN.nc" # IND

ds_A = xr.open_dataset(sim_directory)
ds_B = xr.open_dataset("/home/hplaas/nobackup/JupyterLinks/notebooks/cesm_files/OCNFRAC_only.nc")
ds_B = ds_B.mean(dim="time")
OCNFRAC = ds_B["OCNFRAC"]  

# EXTRACT VARIABLES ---------------------------------------------------------------
selected_vars = ["FEANSOLDRY", "FEANSOLWET", "FEANTOTDRY", "FEANTOTWET",
                 "FEBBSOLDRY", "FEBBSOLWET", "FEBBTOTDRY", "FEBBTOTWET",
                 "FEDUSOLDRY", "FEDUSOLWET", "FEDUTOTDRY", "FEDUTOTWET", 
#                 "FESOLDRY", "FESOLWET", "FETOTDRY", "FETOTWET", 
                 ]  

all_FE_DEP_1 = ds_A[selected_vars]

# TOGGLE DUST SCALING ON FOR PD RUNS
#all_FE_DEP_1['FEDUSOLDRY'] = all_FE_DEP_1['FEDUSOLDRY']*3.8
#all_FE_DEP_1['FEDUTOTDRY'] = all_FE_DEP_1['FEDUTOTDRY']*3.8
#all_FE_DEP_1['FEDUSOLWET'] = all_FE_DEP_1['FEDUSOLWET']*3.8
#all_FE_DEP_1['FEDUTOTWET'] = all_FE_DEP_1['FEDUTOTWET']*3.8

all_FE_DEP_1 = all_FE_DEP_1.mean(dim="time")

FEANTOTDEP = all_FE_DEP_1['FEANTOTDRY'] + all_FE_DEP_1['FEANTOTWET'] 
FEANSOLDEP = all_FE_DEP_1['FEANSOLDRY'] + all_FE_DEP_1['FEANSOLWET'] 

FEBBTOTDEP = all_FE_DEP_1['FEBBTOTDRY'] + all_FE_DEP_1['FEBBTOTWET'] 
FEBBSOLDEP = all_FE_DEP_1['FEBBSOLDRY'] + all_FE_DEP_1['FEBBSOLWET'] 

FEDUTOTDEP = all_FE_DEP_1['FEDUTOTDRY'] + all_FE_DEP_1['FEDUTOTWET']
FEDUSOLDEP = all_FE_DEP_1['FEDUSOLDRY'] + all_FE_DEP_1['FEDUSOLWET']

FETOTDEP = FEANTOTDEP + FEBBTOTDEP + FEDUTOTDEP
FESOLDEP = FEANSOLDEP + FEBBSOLDEP + FEDUSOLDEP

all_FE_DEP = xr.Dataset({
    # TOTAL SOURCES
    "FEANSOLDEP_mean": FEANSOLDEP,
    "FEBBSOLDEP_mean": FEBBSOLDEP,
    "FEDUSOLDEP_mean": FEDUSOLDEP,
    "FEANTOTDEP_mean": FEANTOTDEP,
    "FEBBTOTDEP_mean": FEBBTOTDEP,
    "FEDUTOTDEP_mean": FEDUTOTDEP,
    "FESOLDEP_mean": FESOLDEP,
    "FETOTDEP_mean": FETOTDEP,

    "OCNFRAC": OCNFRAC,
    
    "lat": all_FE_DEP_1['lat'],
    "lon": all_FE_DEP_1['lon']
    })

# FIND AREA OF INDIVIDUAL GRID CELLS --------------------------------
lon = all_FE_DEP['lon'].values  # Longitude in degrees
lat = all_FE_DEP['lat'].values  # Latitude in degrees

lon_rads = np.radians(lon)
lat_rads = np.radians(lat)
d_lat = np.abs(lat[1] - lat[0])  # Latitude grid spacing in degrees
d_lon = np.abs(lon[1] - lon[0])  # Longitude grid spacing in degrees
g_lat = np.radians(d_lat / 2)  # Latitude half-spacing in radians
g_lon = np.radians(d_lon / 2)  # Longitude half-spacing in radians

R = 6.3781E6 
cell_areas_staggered = []
for i in range(len(lat)):
    for j in range(len(lon)):
        lat_center = lat_rads[i]
        lon_center = lon_rads[j]
        lat_north = lat_center + g_lat
        lat_south = lat_center - g_lat
        lat_north = np.clip(lat_north, -np.pi / 2, np.pi / 2)
        lat_south = np.clip(lat_south, -np.pi / 2, np.pi / 2)
        area = R**2 * (np.sin(lat_north) - np.sin(lat_south)) * (2 * g_lon)
        cell_areas_staggered.append(area)

cell_areas_staggered = np.array(cell_areas_staggered).reshape(len(lat), len(lon))

# Verify to see if areas add to 5.1E14 in m^2
sum_sa_earth = cell_areas_staggered.sum()
print(f"surface area, {sum_sa_earth:3e}") 

# add cell area to original xarrays for easy calling 
all_FE_DEP['cell_area'] = xr.DataArray(
    cell_areas_staggered,
    dims=["lat", "lon"],  # Same dimensions as in the original dataset
    coords={"lat": all_FE_DEP['lat'], "lon": all_FE_DEP['lon']},  # Use original coordinates
    attrs={
        "units": "m^2",  # Specify units for the cell area
        "description": "Calculated grid cell area using staggered grid approach",
    },
)

# ---- ASSIGNING ATM. DEPOSITION REGIONS 
import pandas as pd
# Dictionaries to store both individual and total budgets
global_budgets = {}
individual_cell_emissions = {}

# Define regional conditions
# DO NOT CHANGE THESE!!! THEY ARE GOOD NOW FOR ADDING FINAL BUDGETS TO TOTAL 
# South Atlantic
SATL_condition = (all_FE_DEP['lon'] >= 295.0) & (all_FE_DEP['lat'] <= -15.0) & (all_FE_DEP['lat'] > -48.0) | \
                 (all_FE_DEP['lon'] < 30) & (all_FE_DEP['lat'] <= -15.0) & (all_FE_DEP['lat'] >-48.0)
# North Atlantic
NATL_condition = (all_FE_DEP['lon'] > 265.0) & (all_FE_DEP['lat'] > 29.0) & (all_FE_DEP['lat'] < 60.0) | \
                 (all_FE_DEP['lon'] < 110.0) & (all_FE_DEP['lat'] > 29.0) & (all_FE_DEP['lat'] < 60.0)

# Gulf of Mexico
#GOM_condition = (all_FE_DEP['lon'] > 262.0) & (all_FE_DEP['lon'] <= 280.0) & \
#                (all_FE_DEP['lat'] > 15.0) & (all_FE_DEP['lat'] <= 35.0)

# Arabian Sea
AS_condition = (all_FE_DEP['lon'] < 78.0) & (all_FE_DEP['lon'] >= 30.0) & \
               (all_FE_DEP['lat'] <= 29.0) & (all_FE_DEP['lat'] > -48.0)
# Bay of Bengal
BB_condition = (all_FE_DEP['lon'] < 110.0) & (all_FE_DEP['lon'] >= 78.0) & \
               (all_FE_DEP['lat'] <= 29.0) & (all_FE_DEP['lat'] > -48.0)
# Indian Ocean 
INDO_condition = (all_FE_DEP['lon'] < 110.0) & (all_FE_DEP['lon'] >= 30.0) & \
               (all_FE_DEP['lat'] <= 29.0) & (all_FE_DEP['lat'] > -48.0)
# Southeastern Asia
SEAS_condition = (all_FE_DEP['lon'] < 150.0) & (all_FE_DEP['lon'] >= 110.0) & \
               (all_FE_DEP['lat'] < 60.0) & (all_FE_DEP['lat'] > -15.0)
# ENorth Pacific
ENPAC_condition = (all_FE_DEP['lon'] <= 180.0) & (all_FE_DEP['lon'] >= 150.0) & \
               (all_FE_DEP['lat'] < 60.0) & (all_FE_DEP['lat'] > 29.0)
# WNorth Pacific
WNPAC_condition = (all_FE_DEP['lon'] <= 265.0) & (all_FE_DEP['lon'] >= 180.0) & \
               (all_FE_DEP['lat'] < 60.0) & (all_FE_DEP['lat'] > 29.0)
# North Pacific
NPAC_condition = (all_FE_DEP['lon'] <= 265.0) & (all_FE_DEP['lon'] >= 150.0) & \
               (all_FE_DEP['lat'] < 60.0) & (all_FE_DEP['lat'] > 29.0)
# Arctic
ARCT_condition = (all_FE_DEP['lon'] >= 0.0) & (all_FE_DEP['lat'] >= 60.0) &\
               (all_FE_DEP['lon'] >= 0.0)  & (all_FE_DEP['lat'] <= 90.0)
# Australia/South Pacific
AUSP_condition = (all_FE_DEP['lon'] < 295.0) & (all_FE_DEP['lon'] >= 110.0) & \
               (all_FE_DEP['lat'] <= -15.0) & (all_FE_DEP['lat'] > -48.0)
# Southern Ocean
SO_condition = (all_FE_DEP['lon'] >= 0.0) & (all_FE_DEP['lat'] <= -48.0) &\
               (all_FE_DEP['lon'] >= 0.0)  & (all_FE_DEP['lat'] >= -90.0)
# Central Pacific / Asia
CPAO_condition = (all_FE_DEP['lon'] >= 150.0) & (all_FE_DEP['lat'] > -15.0) & (all_FE_DEP['lat'] <= 29.0) | \
                 (all_FE_DEP['lon'] < 30.0) & (all_FE_DEP['lat'] > -15.0) & (all_FE_DEP['lat'] <= 29.0)

# Initialize an empty dictionary to store the grid cell counts
total_cell_counts = {}
total_cell_counts['total'] = all_FE_DEP.dims['lat'] * all_FE_DEP.dims['lon']
total_grid_cells = total_cell_counts['total'] 

# Define Ocean_condition as a boolean mask, not lat/lon operations
Ocean_condition = all_FE_DEP["OCNFRAC"] >= 0.5  

# Initialize storage dictionaries
individual_cell_emissions = {}
global_budgets = {}

# --------- EMISSIONS BUDGETS
# Loop over all variables in the dataset
for var_name in all_FE_DEP.data_vars:
    if "FE" in var_name:  
        # Calculate individual emissions for each grid cell and convert units from kg to Gg
        individual_cell_emissions[var_name] = (
            all_FE_DEP[var_name] * all_FE_DEP['cell_area'] * 3600 * 24 * 365 * 1E-6
        )

        # Calculate the total budget by summing individual emissions
        total_budget = float(individual_cell_emissions[var_name].sum().values)  
        # Calculate Regional budgets by summing specified grid cells
        SATL_budget = float(individual_cell_emissions[var_name].where(SATL_condition).sum().values)
        NATL_budget = float(individual_cell_emissions[var_name].where(NATL_condition).sum().values)
#        GOM_budget = float(individual_cell_emissions[var_name].where(GOM_condition).sum().values)
        AS_budget = float(individual_cell_emissions[var_name].where(AS_condition).sum().values)
        BB_budget = float(individual_cell_emissions[var_name].where(BB_condition).sum().values)
        INDO_budget = float(individual_cell_emissions[var_name].where(INDO_condition).sum().values)
        SEAS_budget = float(individual_cell_emissions[var_name].where(SEAS_condition).sum().values)
        NPAC_budget = float(individual_cell_emissions[var_name].where(NPAC_condition).sum().values)
        ENPAC_budget = float(individual_cell_emissions[var_name].where(ENPAC_condition).sum().values)
        WNPAC_budget = float(individual_cell_emissions[var_name].where(WNPAC_condition).sum().values)
        ARCT_budget = float(individual_cell_emissions[var_name].where(ARCT_condition).sum().values)
        AUSP_budget = float(individual_cell_emissions[var_name].where(AUSP_condition).sum().values)
        SO_budget = float(individual_cell_emissions[var_name].where(SO_condition).sum().values)
        CPAO_budget = float(individual_cell_emissions[var_name].where(CPAO_condition).sum().values)
        
        # Ocean grid cells only
        Ocean_budget = float(individual_cell_emissions[var_name].where(Ocean_condition).sum().values)
        SATL_budget_oc = float(individual_cell_emissions[var_name].where(SATL_condition & Ocean_condition).sum().values)
        NATL_budget_oc = float(individual_cell_emissions[var_name].where(NATL_condition & Ocean_condition).sum().values)
#        GOM_budget_oc = float(individual_cell_emissions[var_name].where(GOM_condition & Ocean_condition).sum().values)
        AS_budget_oc = float(individual_cell_emissions[var_name].where(AS_condition & Ocean_condition).sum().values)
        BB_budget_oc = float(individual_cell_emissions[var_name].where(BB_condition & Ocean_condition).sum().values)
        INDO_budget_oc = float(individual_cell_emissions[var_name].where(INDO_condition & Ocean_condition).sum().values)
        SEAS_budget_oc = float(individual_cell_emissions[var_name].where(SEAS_condition & Ocean_condition).sum().values)
        NPAC_budget_oc = float(individual_cell_emissions[var_name].where(NPAC_condition & Ocean_condition).sum().values)
        ENPAC_budget_oc = float(individual_cell_emissions[var_name].where(ENPAC_condition & Ocean_condition).sum().values)
        WNPAC_budget_oc = float(individual_cell_emissions[var_name].where(WNPAC_condition & Ocean_condition).sum().values)
        ARCT_budget_oc = float(individual_cell_emissions[var_name].where(ARCT_condition & Ocean_condition).sum().values)
        AUSP_budget_oc = float(individual_cell_emissions[var_name].where(AUSP_condition & Ocean_condition).sum().values)
        SO_budget_oc = float(individual_cell_emissions[var_name].where(SO_condition & Ocean_condition).sum().values)
        CPAO_budget_oc = float(individual_cell_emissions[var_name].where(CPAO_condition & Ocean_condition).sum().values)
  
        # Store the budgets for the variable in a nested dictionary
        global_budgets[var_name] = {
            "Total_Budget": total_budget,
            "SATL_Budget": SATL_budget,
            "NATL_Budget": NATL_budget,
#            "GOM_Budget": GOM_budget,
            "AS_Budget": AS_budget,
            "BB_Budget": BB_budget,
            "INDO_budget": INDO_budget,
            "SEAS_Budget": SEAS_budget,
            "NPAC_Budget": NPAC_budget,
            "ENPAC_Budget": ENPAC_budget,
            "WNPAC_Budget": WNPAC_budget,
            "ARCT_Budget": ARCT_budget,
            "AUSP_Budget": AUSP_budget,
            "SO_Budget": SO_budget,
            "CPAO_Budget": CPAO_budget,

            "Ocean_budget": Ocean_budget,
            "SATL_Budget_oc": SATL_budget_oc,
            "NATL_Budget_oc": NATL_budget_oc,
#            "GOM_Budget_oc": GOM_budget_oc,
            "AS_Budget_oc": AS_budget_oc,
            "BB_Budget_oc": BB_budget_oc,
            "INDO_budget_oc": INDO_budget_oc,
            "SEAS_Budget_oc": SEAS_budget_oc,
            "NPAC_Budget_oc": NPAC_budget_oc,
            "ENPAC_Budget_oc": ENPAC_budget_oc,
            "WNPAC_Budget_oc": WNPAC_budget_oc,
            "ARCT_Budget_oc": ARCT_budget_oc,
            "AUSP_Budget_oc": AUSP_budget_oc,
            "SO_Budget_oc": SO_budget_oc,
            "CPAO_Budget_oc": CPAO_budget_oc
        }

budget_df = pd.DataFrame(global_budgets).T  # Transpose to get variables as rows and budgets as columns

# Reset index and give the proper column name for variables
budget_df.reset_index(inplace=True)
budget_df.rename(columns={'index': 'Variable'}, inplace=True)

# Ensure all columns have the correct numeric type
budget_df = budget_df.apply(pd.to_numeric, errors='ignore')  # Apply to convert all numeric
budget_df = budget_df.round(3)

# Save the total budgets and individual emissions to separate sheets in an Excel file
budget_df.to_csv(f"/home/hplaas/nobackup/JupyterLinks/notebooks/cesm_files/output_files/deposition_budgets/FeDeposition_Budgets_{case}_MED_DUSTFETUNED.csv", index=False)

print("done") 

surface area, 5.112020e+14


/gpfsm/dnb33/tdirs/batch/slurm.55829545.hplaas/ipykernel_223122/615004060.py:177: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  total_cell_counts['total'] = all_FE_DEP.dims['lat'] * all_FE_DEP.dims['lon']


done


/gpfsm/dnb33/tdirs/batch/slurm.55829545.hplaas/ipykernel_223122/615004060.py:273: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  budget_df = budget_df.apply(pd.to_numeric, errors='ignore')  # Apply to convert all numeric
